# Notebook of comparison between Abromics Kg powered by SOSA or RDF data cube as core ontologies

In [1]:
# imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [2]:
# Load the local ABRomics report
with open("reports.json", "r") as f:
    reports = json.load(f)

## Benchmark utility class 

The benchmark class is a utility that allows to perform query benchmarks in bulk for a specific graph and a set of queries.
The result of the benchmark can be compared using this utility class as the process is the same whatever the graph and the queries are.

### Parameters

- **graph**     : RDFlib graph object in which the benchmark should be performed
- **queries**   : A list of queries and queries description that should be run in bulk in the benchmark

```python
queries = [
    {
        "description": "description of the query",
        "content": "content of the query itself"
    }
]
```

In [3]:
## Benchmark class utility class that allows to have all the tools to benchmark graphs and queries

class Benchmark:
    
    def __init__(self, graph, queries):
        self.graph = graph
        self.queries = queries
        self.benchmarkData = [[], [], [], []]
        self.benchmarkRowNames = ["description", "avg_elapsed_time (ms)", "max_elapsed_time (ms)", "min_elapse_time (ms)"]
        self.benchmarkDf = pd.DataFrame()
    
    def launch(self):
        for query in queries:
            times = []
            for i in range(0, 100):
                startTime = time.time()
                res = self.graph.query(query["content"])
                endTime = time.time() - startTime
                times.append(round(endTime * 1000, 2))
            self.benchmarkData[0].append(query["description"])
            self.benchmarkData[1].append(round(sum(times) / len(times), 2))
            self.benchmarkData[2].append(max(times))
            self.benchmarkData[3].append(min(times))
        self.benchmarkDf = pd.DataFrame(self.benchmarkData, index=self.benchmarkRowNames, columns=[f"query {i}" for i in range(1, len(self.queries) + 1)])
        
    def displayResults(self):
        print(self.benchmarkDf)
        


## Create an ABRomics graph using SOSA as the core ontology 

Core ontology : 
- [SOSA / SSN](https://www.w3.org/TR/vocab-ssn/)

Specific ontologies :
- [NCIT](https://bioportal.bioontology.org/ontologies/NCIT)
- [OHMI](https://bioportal.bioontology.org/ontologies/OHMI)
- [ARO](https://bioportal.bioontology.org/ontologies/ARO)
- [GO](https://bioportal.bioontology.org/ontologies/GO)

In [4]:
g = Graph()
    
# Placeholder namespaces
ABROMICS = Namespace("http://www.abromics.org/ontology/")
SOSA = Namespace('http://www.w3.org/ns/sosa/')
NCIt = Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl/')
OHMI = Namespace('https://bioportal.bioontology.org/ontologies/OHMI/')
ARO = Namespace('https://bioportal.bioontology.org/ontologies/ARO/')
GO = Namespace('https://bioportal.bioontology.org/ontologies/GO/')

## Namespaces used in Uniprot
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
UP = Namespace("http://purl.uniprot.org/core/")


## Sample metadata
sampleMetadata = reports[0]["sections"][0]["data"][0]["values"]
sampleMetadata

## Storing the sample
sampleId = uuid.uuid1()
g.add((ABROMICS[str(sampleId)], RDF.type, SOSA.ObservableProperty))
g.add((ABROMICS[str(sampleId)], ABROMICS.MicroOrganism, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][1]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.CollectedAt, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][2])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Source, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][3])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SampleType, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][4])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Host, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][5]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Location, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][6])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingTechnology, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][7]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingPlatform, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][8]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterName, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][9]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterEmail, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][10]).replace(' ', '_')]))

## Storing the analysis
analysisId = uuid.uuid1()
g.add((ABROMICS[str(analysisId)], RDF.type, SOSA.Observation))
g.add((ABROMICS[str(analysisId)], SOSA.ObservedProperty, ABROMICS[str(sampleId)]))

## Storing the FeatureOfInterest
featureOfInterestId = uuid.uuid1()
g.add((ABROMICS[str(featureOfInterestId)], SOSA.isFeatureOfInterest, ABROMICS[str(analysisId)]))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, SOSA.FeatureOfInterest))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, ABROMICS.Genomic))

## Storing the Microbiome
microbiomeId = uuid.uuid1()
g.add((ABROMICS[str(microbiomeId)], RDF.type, OHMI.HumanMicrobiome))
g.add((ABROMICS[str(featureOfInterestId)], ABROMICS.has, ABROMICS[str(microbiomeId)]))

## Store a single strain observed in the sample
strainId = uuid.uuid1()
g.add((ABROMICS[str(strainId)], RDF.type, ARO.Strain))
g.add((ABROMICS[str(microbiomeId)], ABROMICS.contains, ABROMICS[str(strainId)]))
g.add((ABROMICS[str(strainId)], ABROMICS.MLSTProfile, Literal("MLST_profile", datatype=XSD.string)))
g.add((ABROMICS[str(strainId)], ABROMICS.STProfile, Literal("ST_profile", datatype=XSD.string)))

## Store a molecule the strain is resistant to
moleculeId = uuid.uuid1()
g.add((ABROMICS[str(moleculeId)], RDF.type, ARO.Molecule))
g.add((ABROMICS[str(strainId)], ABROMICS.isResistantTo, ABROMICS[str(moleculeId)]))
g.add((ABROMICS[str(moleculeId)], RDFS.label, Literal(str(reports[0]["sections"][2]["data"][0]["values"][8][0]), datatype=XSD.string))) ## store the name of the molecule


## Store the resistance gene associated with the molecule the bacteria is resistant to
geneId = uuid.uuid1()
g.add((ABROMICS[str(geneId)], RDF.type, UP.Gene))
g.add((ABROMICS[str(geneId)], SKOS.prefName, Literal("aadA13", datatype=XSD.string)))
g.add((ABROMICS[str(geneId)], ABROMICS.givesResistanceTo, ABROMICS[str(moleculeId)]))


g.serialize(destination="antibiotic-resistance-sosa.ttl", format='ttl')
    
g

<Graph identifier=N7eb5c5fb3978489fb6fdbc6d849f1d0f (<class 'rdflib.graph.Graph'>)>

In [5]:
## Query 1 :
query1 = """
    PREFIX ns1: <http://www.abromics.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX aro: <https://bioportal.bioontology.org/ontologies/ARO/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX up: <http://purl.uniprot.org/core/>

    SELECT ?geneName ?moleculeName
    WHERE {
        ?gene a up:Gene .
        ?gene skos:prefName ?geneName .
        ?molecule a aro:Molecule .
        ?gene ns1:givesResistanceTo ?molecule .
        ?molecule rdfs:label ?moleculeName .
    }
"""

query2 = """
    SELECT ?s ?p ?o 
    WHERE {
        ?s ?p ?o .
    }
"""


queries = [
    {
        "description": "Get all the resistance genes and the resistance molecule",
        "content": query1
    },
    {
        "description": "all the nodes in the graph",
        "content": query2
    }
]

In [6]:
## Using benchmark class with SOSA graph and a query

benchmark = Benchmark(g, queries)

benchmark.launch()
benchmark.displayResults()

                                                                 query 1  \
description            Get all the resistance genes and the resistanc...   
avg_elapsed_time (ms)                                               4.65   
max_elapsed_time (ms)                                             135.86   
min_elapse_time (ms)                                                3.17   

                                          query 2  
description            all the nodes in the graph  
avg_elapsed_time (ms)                        1.55  
max_elapsed_time (ms)                         1.9  
min_elapse_time (ms)                         1.48  


## Create an ABRomics graph using RDF Datacube as the core ontology

Core ontology:
- [RDF Data Cube](https://www.w3.org/TR/vocab-data-cube/)

Insipired by the graph example presented in section 6 of the RDF data cube documentation



In [22]:
g = Graph()

ABROMICS = Namespace("http://www.abromics.org/ontology/")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")
DCT = Namespace("http://purl.org/dc/terms/")

QB = Namespace("http://purl.org/linked-data/cube#")

# Read part 6 of the RDF data cube documentation to understand how such data should be encoded

analysisResult = pd.DataFrame(reports[0]["sections"][2]["data"][0]["values"], index=reports[0]["sections"][2]["data"][0]["header"])

# Create all the dimensions and measures
resistanceGenePropertyId = uuid.uuid1()
g.add((ABROMICS[str(resistanceGenePropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(resistanceGenePropertyId)], RDFS.label, Literal("Resistance Gene", datatype=XSD.string)))

targetAntibioticPropertyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDFS.label, Literal("Target Antibiotic", datatype=XSD.string)))


# Create slices
targetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(targetAntibioticSliceId)], RDFS.label, Literal("slice by antibiotic target", datatype=XSD.string)))


# Create slice key
targetAntibioticSliceKeyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDF.type, QB.SliceKey))
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDFS.label, Literal("slice by target antibiotic", datatype=XSD.string)))


# Create concrete slice (the real slice entity that will be linked to a dataset)
concreteTargetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.sliceStructure, ABROMICS[str(targetAntibioticSliceId)]))

      
# Create the component specification 
componentSpecificationId = uuid.uuid1()
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(resistanceGenePropertyId)]))
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(targetAntibioticPropertyId)]))
      

# Create the Dataset Structure
datasetStructureId = uuid.uuid1()
g.add((ABROMICS[str(datasetStructureId)], RDF.type, QB.DataStructureDefinition))
g.add((ABROMICS[str(datasetStructureId)], QB.Component, ABROMICS[str(componentSpecificationId)]))


# Create the dataset
datasetId = uuid.uuid1()
g.add((ABROMICS[str(datasetId)], RDF.type, QB.DataSet))
g.add((ABROMICS[str(datasetId)], DCT.title, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.label, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.comment, Literal("Comments about the dataset", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], QB.sliceKey, ABROMICS[str(targetAntibioticSliceKeyId)]))
g.add((ABROMICS[str(datasetId)], QB.slice, ABROMICS[str(concreteTargetAntibioticSliceId)]))


# Creation of all the observations of the two rows (Resistance gene and target antibiotic)
for gene in reports[0]["sections"][2]["data"][0]["values"][0]:
    geneObservationId = uuid.uuid1()
    g.add((ABROMICS[str(geneObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(geneObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(geneObservationId)], RDFS.label, Literal(gene, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(geneObservationId)]))

for antibiotic in reports[0]["sections"][2]["data"][0]["values"][8]:
    antibioticObservationId = uuid.uuid1()
    g.add((ABROMICS[str(antibioticObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(antibioticObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(antibioticObservationId)], RDFS.label, Literal(antibiotic, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(antibioticObservationId)]))

g.serialize(destination="antibiotic-resistance-qb.ttl", format='ttl')

g

# Checkout the handmade draft on the desk to make a decision on the structure of the KG using RDF data cube

<Graph identifier=Nb4ee51aa62ba42d9a7cfe5d2101bdc7e (<class 'rdflib.graph.Graph'>)>

In [24]:
## Query 1 :
query1 = """
    PREFIX ns1: <http://www.abromics.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX dct: <http://purl.org/dc/terms/>
    PREFIX qb: <http://purl.org/linked-data/cube#>
    

    SELECT ?geneNames ?moleculeNames
    WHERE {
        ?dataset a qb:DataSet . 
        ?geneNames a qb:Observation ;
            rdfs:label "Target antibiotic" .
        ?moleculeNames a qb:Observation ; 
            rdfs:label "Resistance gene"
    }
"""

query2 = """
    SELECT ?s ?p ?o 
    WHERE {
        ?s ?p ?o .
    }
"""


queries = [
    {
        "description": "Get all the resistance genes and the resistance molecule",
        "content": query1
    },
    {
        "description": "all the nodes in the graph",
        "content": query2
    }
]

In [26]:
## Using benchmark class with RDF data cube graph and a query

benchmark = Benchmark(g, queries)

benchmark.launch()
benchmark.displayResults()

                                                                 query 1  \
description            Get all the resistance genes and the resistanc...   
avg_elapsed_time (ms)                                               3.25   
max_elapsed_time (ms)                                              10.29   
min_elapse_time (ms)                                                2.94   

                                          query 2  
description            all the nodes in the graph  
avg_elapsed_time (ms)                        1.64  
max_elapsed_time (ms)                         2.6  
min_elapse_time (ms)                         1.49  


## Differences between SOSA and RDF data cube

From a performance perspective, querying a graph using SOSA or RDF data cube is similar, slighly faster for RDF data cube (1ms faster) when not using any slices in the dataset.
However SOSA is a bit easier to query as the entities to manipulate such as Observation, Sample and ObservableProperties are much more concrete than the sometimes confusing ComponentProperties, Slices, SliceKeys
of RDF data cube.

With RDF data cube the structure of the data needs to be well understood by the user before performing a query, even more if the dataset have to be sliced during the query. 

## Conclusion

From a ease to use point of view, SOSA seems to fit the ABRomics result report much better than RDF data cube. But in term of performance, it's yet not possible to know which ontology is more suitable.
Further investigation about dataset slice have to be done for RDF data cube. More complex queries need to be performed on both graphs to compare performance objectively. 